# Cell 1 Views

In [1]:
%%sql
CREATE OR REPLACE VIEW gold.v_in_gps AS
SELECT event_id, trip_id, vehicle_id, event_time, ingestion_time, dq_flags, latitude, longitude, speed_kmh, gps_accuracy_meters
FROM silver.silver_gps;

CREATE OR REPLACE VIEW gold.v_in_engine AS
SELECT event_id, trip_id, vehicle_id, event_time, ingestion_time, dq_flags, engine_on, engine_rpm, engine_temperature_c, vehicle_speed_kmh
FROM silver.silver_engine;

CREATE OR REPLACE VIEW gold.v_in_fuel AS
SELECT event_id, trip_id, vehicle_id, event_time, ingestion_time, dq_flags, fuel_level_liters, trip_fuel_used_liters, odometer_km
FROM silver.silver_fuel;

CREATE OR REPLACE VIEW gold.v_in_temperature AS
SELECT event_id, trip_id, vehicle_id, event_time, ingestion_time, dq_flags, temperature_value AS temperature_c, sensor_status
FROM silver.silver_temperature;

StatementMeta(, a28a30e1-c1ac-4b6d-b217-1c2a6a75becb, 8, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

# Table DDL (this is the contract)

In [2]:
%%sql
CREATE TABLE IF NOT EXISTS gold.dim_vehicle (
  vehicle_key BIGINT, vehicle_id STRING, registration_number STRING,
  vehicle_type STRING, refrigeration_capable BOOLEAN, device_id STRING,
  fuel_type STRING, fuel_tank_capacity_liters DOUBLE, model_year INT,
  active_from TIMESTAMP, active_to TIMESTAMP,
  effective_from TIMESTAMP, effective_to TIMESTAMP, is_current BOOLEAN,
  type2_hash STRING, row_hash STRING
) USING DELTA;

CREATE TABLE IF NOT EXISTS gold.dim_driver (
  driver_key BIGINT, driver_id STRING, driver_name STRING, license_category STRING,
  hire_date DATE, active_from TIMESTAMP, active_to TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS gold.dim_route (
  route_key BIGINT, route_id STRING, route_name STRING, origin_name STRING, destination_name STRING,
  planned_distance_km DOUBLE, allowed_deviation_meters INT, expected_duration_minutes INT,
  arrival_grace_minutes INT, speed_limit_kmh DOUBLE, checkpoint_count INT
) USING DELTA;

CREATE TABLE IF NOT EXISTS gold.dim_cargo_type (
  cargo_type_key BIGINT, cargo_type_id STRING, cargo_name STRING, is_temperature_controlled BOOLEAN,
  temperature_min_c DOUBLE, temperature_max_c DOUBLE, cargo_value_per_unit DOUBLE, currency_code STRING,
  max_excursion_minutes INT, spoilage_loss_pct DOUBLE, alert_hold_minutes INT
) USING DELTA;

CREATE TABLE IF NOT EXISTS gold.dim_date (
  date_key INT, date DATE, year INT, quarter INT, month INT, month_name STRING,
  day INT, day_of_week INT, day_name STRING, is_weekend BOOLEAN
) USING DELTA;

CREATE TABLE IF NOT EXISTS gold.fact_trip_performance (
  trip_id STRING,
  vehicle_key BIGINT, driver_key BIGINT, route_key BIGINT, cargo_type_key BIGINT, planned_start_date_key INT,
  vehicle_id STRING, driver_id STRING, route_id STRING, shipment_id STRING, cargo_type_id STRING,
  planned_start_time TIMESTAMP, planned_end_time TIMESTAMP, cargo_quantity DOUBLE, trip_status STRING,
  actual_start_time TIMESTAMP, actual_end_time TIMESTAMP, arrived BOOLEAN, arrival_status STRING, late_min DOUBLE,
  distance_km DOUBLE, fuel_used_l DOUBLE, fuel_l_per_100km DOUBLE, fuel_counter_anomalies BIGINT,
  engine_on_min DOUBLE, driving_min DOUBLE, idle_min_total DOUBLE, excessive_idle_min DOUBLE,
  speeding_min DOUBLE, deviation_episodes BIGINT,
  score_speeding_penalty DOUBLE, score_idling_penalty DOUBLE, score_deviation_penalty DOUBLE,
  score_lateness_penalty DOUBLE, trip_score DOUBLE, score_complete BOOLEAN,
  invalid_assignment BOOLEAN, has_telemetry BOOLEAN,
  gps_readings BIGINT, engine_readings BIGINT, fuel_readings BIGINT,
  first_event_time TIMESTAMP, last_event_time TIMESTAMP, built_at TIMESTAMP
) USING DELTA;

CREATE TABLE  IF NOT EXISTS gold.fact_cold_chain (
  trip_id STRING, shipment_id STRING,
  vehicle_key BIGINT, driver_key BIGINT, route_key BIGINT, cargo_type_key BIGINT, planned_start_date_key INT,
  vehicle_id STRING, driver_id STRING, route_id STRING, cargo_type_id STRING,
  measured_min DOUBLE, in_range_min DOUBLE, out_range_min DOUBLE, unmeasured_min DOUBLE,
  min_temperature_c DOUBLE, max_temperature_c DOUBLE,
  excursion_episodes BIGINT, breach_episodes BIGINT, longest_excursion_min DOUBLE,
  breach_flag BOOLEAN, compromised_flag BOOLEAN, cargo_value DOUBLE, cargo_at_risk DOUBLE,
  suspect_readings BIGINT, temp_readings BIGINT, built_at TIMESTAMP
) USING DELTA;

StatementMeta(, a28a30e1-c1ac-4b6d-b217-1c2a6a75becb, 17, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

# Cell 3 — setup and the simple dimensions

In [3]:
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")

def overwrite(df, table, cols):
    # no overwriteSchema on purpose: if the columns differ from the DDL, the write FAILS (contract check)
    df.select(*cols).write.format("delta").mode("overwrite").saveAsTable(table)

# dim_driver
d = spark.table("ref.ref_drivers")
overwrite(d.select(F.xxhash64("driver_id").alias("driver_key"), "driver_id", "driver_name", "license_category",
                   "hire_date", "active_from", F.to_timestamp("active_to").alias("active_to")),
          "gold.dim_driver",
          ["driver_key","driver_id","driver_name","license_category","hire_date","active_from","active_to"])

# dim_route
cp_cnt = spark.table("ref.ref_route_checkpoints").groupBy("route_id").agg(F.count("*").cast("int").alias("checkpoint_count"))
r = spark.table("ref.ref_routes").join(cp_cnt, "route_id", "left")
overwrite(r.withColumn("route_key", F.xxhash64("route_id")), "gold.dim_route",
          ["route_key","route_id","route_name","origin_name","destination_name","planned_distance_km",
           "allowed_deviation_meters","expected_duration_minutes","arrival_grace_minutes","speed_limit_kmh","checkpoint_count"])

# dim_cargo_type
c = (spark.table("ref.ref_cargo_types")
     .withColumn("cargo_type_key", F.xxhash64("cargo_type_id"))
     .withColumn("is_temperature_controlled", F.col("temperature_max_c").isNotNull())
     .withColumn("spoilage_loss_pct", F.col("spoilage_loss_pct") / 100.0))
overwrite(c, "gold.dim_cargo_type",
          ["cargo_type_key","cargo_type_id","cargo_name","is_temperature_controlled","temperature_min_c","temperature_max_c",
           "cargo_value_per_unit","currency_code","max_excursion_minutes","spoilage_loss_pct","alert_hold_minutes"])

# dim_date: covers simulated dates from 30 days before the first trip to 30 days after the last one
rng = spark.table("ref.ref_trips").agg(F.min("planned_start_time").alias("a"), F.max("planned_end_time").alias("b")).first()
dd = spark.sql(f"""
  SELECT explode(sequence(date_sub(to_date('{rng.a:%Y-%m-%d}'), 30), date_add(to_date('{rng.b:%Y-%m-%d}'), 30), interval 1 day)) AS date""")
dd = (dd.withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
        .withColumn("year", F.year("date")).withColumn("quarter", F.quarter("date")).withColumn("month", F.month("date"))
        .withColumn("month_name", F.date_format("date", "MMMM")).withColumn("day", F.dayofmonth("date"))
        .withColumn("day_of_week", F.dayofweek("date")).withColumn("day_name", F.date_format("date", "EEEE"))
        .withColumn("is_weekend", F.dayofweek("date").isin(6, 7)))     # Fri/Sat in Egypt; change if you prefer Sat/Sun
overwrite(dd, "gold.dim_date", ["date_key","date","year","quarter","month","month_name","day","day_of_week","day_name","is_weekend"])

StatementMeta(, a28a30e1-c1ac-4b6d-b217-1c2a6a75becb, 23, Finished, Available, Finished, False)

# Cell 3 validation

In [ ]:
spark.table("gold.dim_cargo_type") \
    .filter(
        "spoilage_loss_pct < 0 OR spoilage_loss_pct > 1"
    ) \
    .select("cargo_type_id", "spoilage_loss_pct") \
    .show(truncate=False)

StatementMeta(, a28a30e1-c1ac-4b6d-b217-1c2a6a75becb, 21, Finished, Available, Finished, False)

+-------------+-----------------+
|cargo_type_id|spoilage_loss_pct|
+-------------+-----------------+
|FOOD_CHILLED |25.0             |
|MEDICINE_COLD|100.0            |
|FROZEN_FOOD  |40.0             |
+-------------+-----------------+



In [ ]:
spark.table("gold.dim_cargo_type") \
    .filter("spoilage_loss_pct IS NULL") \
    .select("cargo_type_id", "spoilage_loss_pct") \
    .show(truncate=False)

StatementMeta(, a28a30e1-c1ac-4b6d-b217-1c2a6a75becb, 22, Finished, Available, Finished, False)

+-------------+-----------------+
|cargo_type_id|spoilage_loss_pct|
+-------------+-----------------+
+-------------+-----------------+



In [ ]:
from pyspark.sql import functions as F, Window

# Expected: every dimension has rows; business keys are unique; no route has fewer than 2 checkpoints.
checks = {
    "drivers_nonempty": spark.table("gold.dim_driver").limit(1).count() == 1,
    "routes_nonempty": spark.table("gold.dim_route").limit(1).count() == 1,
    "cargo_nonempty": spark.table("gold.dim_cargo_type").limit(1).count() == 1,
    "dates_nonempty": spark.table("gold.dim_date").limit(1).count() == 1,
    "duplicate_driver_ids": spark.table("gold.dim_driver").count() == spark.table("gold.dim_driver").select("driver_id").distinct().count(),
    "duplicate_route_ids": spark.table("gold.dim_route").count() == spark.table("gold.dim_route").select("route_id").distinct().count(),
    "duplicate_cargo_ids": spark.table("gold.dim_cargo_type").count() == spark.table("gold.dim_cargo_type").select("cargo_type_id").distinct().count(),
    "duplicate_date_keys": spark.table("gold.dim_date").count() == spark.table("gold.dim_date").select("date_key").distinct().count(),
    "routes_have_2plus_checkpoints": spark.table("gold.dim_route").filter(F.col("checkpoint_count") < 2).count() == 0,
    "spoilage_pct_is_fraction": spark.table("gold.dim_cargo_type").filter("spoilage_loss_pct < 0 OR spoilage_loss_pct > 1").count() == 0,
    "dry_goods_not_cold": spark.table("gold.dim_cargo_type").filter("cargo_type_id = 'DRY_GOODS' AND is_temperature_controlled").count() == 0,
}
print(checks)
assert all(checks.values()), f"Cell 1 failed: {[k for k,v in checks.items() if not v]}"

StatementMeta(, a28a30e1-c1ac-4b6d-b217-1c2a6a75becb, 24, Finished, Available, Finished, False)

{'drivers_nonempty': True, 'routes_nonempty': True, 'cargo_nonempty': True, 'dates_nonempty': True, 'duplicate_driver_ids': True, 'duplicate_route_ids': True, 'duplicate_cargo_ids': True, 'duplicate_date_keys': True, 'routes_have_2plus_checkpoints': True, 'spoilage_pct_is_fraction': True, 'dry_goods_not_cold': True}


# Cell 4 — dim_vehicle (SCD)

In [ ]:
raw = spark.table("ref.ref_vehicles_raw")

conflicts = (raw.groupBy("vehicle_id", "updated_at")
    .agg(F.countDistinct(F.concat_ws("|", "registration_number", "vehicle_type", "fuel_type", "fuel_tank_capacity_liters",
                                     "refrigeration_capable", "device_id", "model_year", "active_from", "active_to")).alias("n"))
    .filter("n > 1"))
assert conflicts.count() == 0, "Conflicting vehicle versions (same vehicle_id + updated_at, different content). Fix the source, do not guess."

v = (raw.select(
        "vehicle_id", "registration_number", "vehicle_type", "fuel_type",
        F.col("fuel_tank_capacity_liters").cast("double").alias("fuel_tank_capacity_liters"),
        (F.lower(F.col("refrigeration_capable")) == "true").alias("refrigeration_capable"),
        "device_id", F.col("model_year").cast("int").alias("model_year"),
        F.to_timestamp("active_from").alias("active_from"),
        F.when(F.upper(F.trim(F.col("active_to"))).isin("NULL", ""), F.lit(None).cast("timestamp"))
         .otherwise(F.to_timestamp("active_to")).alias("active_to"),
        F.to_timestamp("updated_at").alias("updated_at"))
     .dropDuplicates(["vehicle_id", "updated_at"]))      # a file loaded twice must not create versions

# Type 2 columns -> a hash; a new version exists only when the hash changes
v = v.withColumn("type2_hash", F.sha2(F.concat_ws("|", "vehicle_type",
                                   F.col("refrigeration_capable").cast("string"), "device_id"), 256))
w = Window.partitionBy("vehicle_id").orderBy("updated_at")
v = v.withColumn("prev_hash", F.lag("type2_hash").over(w))

ver = v.filter(F.col("prev_hash").isNull() | (F.col("prev_hash") != F.col("type2_hash")))
ver = ver.withColumn("effective_from",
        F.when(F.col("prev_hash").isNull(), F.least("active_from", "updated_at")).otherwise(F.col("updated_at")))
w2 = Window.partitionBy("vehicle_id").orderBy("effective_from")
FOREVER = F.lit("9999-12-31 00:00:00").cast("timestamp")
ver = (ver.withColumn("effective_to", F.coalesce(F.lead("effective_from").over(w2), FOREVER))
          .withColumn("is_current", F.col("effective_to") == FOREVER)
          .select("vehicle_id", "vehicle_type", "refrigeration_capable", "device_id", "type2_hash",
                  "updated_at", "effective_from", "effective_to", "is_current"))

# Type 1 columns: the LATEST row's values are written on every version (a correction shows everywhere)
latest = (v.withColumn("rn", F.row_number().over(Window.partitionBy("vehicle_id").orderBy(F.col("updated_at").desc())))
           .filter("rn = 1")
           .select("vehicle_id", "registration_number", "fuel_type", "fuel_tank_capacity_liters",
                   "model_year", "active_from", "active_to"))

stage = (ver.join(latest, "vehicle_id")
    .withColumn("vehicle_key", F.xxhash64("vehicle_id", F.date_format("updated_at", "yyyy-MM-dd HH:mm:ss")))
    .withColumn("row_hash", F.sha2(F.concat_ws("|",
        F.col("registration_number"), F.col("vehicle_type"), F.col("refrigeration_capable").cast("string"),
        F.col("device_id"), F.col("fuel_type"), F.col("fuel_tank_capacity_liters").cast("string"),
        F.col("model_year").cast("string"), F.col("active_from").cast("string"), F.col("active_to").cast("string"),
        F.col("effective_from").cast("string"), F.col("effective_to").cast("string"),
        F.col("is_current").cast("string")), 256))
    .select("vehicle_key","vehicle_id","registration_number","vehicle_type","refrigeration_capable","device_id",
            "fuel_type","fuel_tank_capacity_liters","model_year","active_from","active_to",
            "effective_from","effective_to","is_current","type2_hash","row_hash"))

stage.createOrReplaceTempView("stg_dim_vehicle")
spark.sql("""
MERGE INTO gold.dim_vehicle AS t
USING stg_dim_vehicle AS s
ON t.vehicle_key = s.vehicle_key
WHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE
""")
spark.sql("SELECT COUNT(*) AS versions, COUNT(DISTINCT vehicle_id) AS vehicles FROM gold.dim_vehicle").show()

StatementMeta(, a28a30e1-c1ac-4b6d-b217-1c2a6a75becb, 26, Finished, Available, Finished, False)

+--------+--------+
|versions|vehicles|
+--------+--------+
|      22|      20|
+--------+--------+



# Cell 4 validation

In [ ]:
from pyspark.sql import functions as F
v = spark.table("gold.dim_vehicle")
current_bad = v.filter(F.col("is_current")).groupBy("vehicle_id").count().filter("count <> 1").count()
key_dupes = v.groupBy("vehicle_key").count().filter("count > 1").count()
overlaps = v.alias("a").join(v.alias("b"),
    (F.col("a.vehicle_id") == F.col("b.vehicle_id")) &
    (F.col("a.vehicle_key") < F.col("b.vehicle_key")) &
    (F.col("a.effective_from") < F.col("b.effective_to")) &
    (F.col("b.effective_from") < F.col("a.effective_to")), "inner").count()
assert v.limit(1).count() == 1, "dim_vehicle is empty"
assert key_dupes == 0, f"Duplicate vehicle_key rows: {key_dupes}"
assert current_bad == 0, f"Vehicles without exactly one current version: {current_bad}"
assert overlaps == 0, f"Overlapping vehicle versions: {overlaps}"
print("Cell 2 passed; versions:", v.count(), "vehicles:", v.select("vehicle_id").distinct().count())

StatementMeta(, a28a30e1-c1ac-4b6d-b217-1c2a6a75becb, 27, Finished, Available, Finished, False)

Cell 2 passed; versions: 22 vehicles: 20


# Cell 5 — settings, helpers, rules

In [ ]:
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")

rules = {r["rule_name"]: float(r["rule_value"]) for r in spark.table("ref.ref_business_rules").collect()}

REQUIRED_RULES = [
 "idle_speed_threshold_kmh", "idle_minutes_threshold", "speeding_tolerance_kmh", "gps_accuracy_max_meters",
 "deviation_consecutive_readings", "gps_jump_max_kmh", "temperature_spike_c", "flatline_readings", "offline_missed_intervals",
 "expected_interval_gps_sec", "expected_interval_engine_sec", "expected_interval_fuel_sec", "expected_interval_temp_sec",
 "score_weight_speeding", "score_weight_idling", "score_weight_deviation", "score_weight_lateness",
 "score_limit_speeding_pct", "score_limit_idling_pct", "score_limit_deviation_episodes", "score_limit_late_minutes"]
missing = [r for r in REQUIRED_RULES if r not in rules]
assert not missing, f"Missing business rules in ref.ref_business_rules: {missing}"   # no silent defaults any more
def rule(name): return rules[name]

IDLE_KMH     = rule("idle_speed_threshold_kmh")
IDLE_MIN     = rule("idle_minutes_threshold")
SPEED_TOL    = rule("speeding_tolerance_kmh")
GPS_ACC_MAX  = rule("gps_accuracy_max_meters")
DEV_READINGS = int(rule("deviation_consecutive_readings"))
JUMP_KMH     = rule("gps_jump_max_kmh")
SPIKE_C      = rule("temperature_spike_c")
FLAT_N       = int(rule("flatline_readings"))
CAP_MULT     = int(rule("offline_missed_intervals"))        # same meaning: 3 missed intervals = a hole in the data
EXP = {"gps": rule("expected_interval_gps_sec"), "engine": rule("expected_interval_engine_sec"),
       "fuel": rule("expected_interval_fuel_sec"), "temperature": rule("expected_interval_temp_sec")}
ARRIVAL_RADIUS_M = rules.get("arrival_radius_m", 1000.0)    # the only allowed default (D9); add the rule to the generator later

W_SP, W_ID, W_DV, W_LT = (rule("score_weight_speeding"), rule("score_weight_idling"),
                          rule("score_weight_deviation"), rule("score_weight_lateness"))
LIM_SP = rule("score_limit_speeding_pct") / 100.0
LIM_ID = rule("score_limit_idling_pct") / 100.0
LIM_DV = rule("score_limit_deviation_episodes")
LIM_LT = rule("score_limit_late_minutes")
print("rules loaded:", len(rules), "| arrival radius:", ARRIVAL_RADIUS_M)

StatementMeta(, a28a30e1-c1ac-4b6d-b217-1c2a6a75becb, 28, Finished, Available, Finished, False)

rules loaded: 32 | arrival radius: 1000.0


# Cell 5 validation

In [ ]:
assert spark.conf.get("spark.sql.session.timeZone") == "UTC"
assert all(x > 0 for x in EXP.values()), f"Expected intervals must be positive: {EXP}"
assert CAP_MULT > 0 and DEV_READINGS >= 1
assert abs(W_SP + W_ID + W_DV + W_LT - 100.0) < 1e-6, "Score weights must sum to 100"
assert LIM_SP > 0 and LIM_ID > 0 and LIM_DV > 0 and LIM_LT > 0
assert FLAT_N >= 2 and SPIKE_C > 0 and JUMP_KMH > 0 and ARRIVAL_RADIUS_M > 0
print("Cell 5 passed. Active rules:", sorted(rules))

StatementMeta(, a28a30e1-c1ac-4b6d-b217-1c2a6a75becb, 30, Finished, Available, Finished, False)

Cell 5 passed. Active rules: ['cargo_temperature_valid_max_c', 'cargo_temperature_valid_min_c', 'deviation_consecutive_readings', 'dq_window_minutes', 'engine_rpm_max', 'engine_temperature_max_c', 'expected_interval_engine_sec', 'expected_interval_fuel_sec', 'expected_interval_gps_sec', 'expected_interval_temp_sec', 'flatline_readings', 'future_timestamp_tolerance_minutes', 'gps_accuracy_max_meters', 'gps_jump_max_kmh', 'idle_minutes_threshold', 'idle_speed_threshold_kmh', 'invalid_rate_alert_pct', 'late_data_watermark_minutes', 'offline_missed_intervals', 'quality_gate_max_invalid_pct', 'score_limit_deviation_episodes', 'score_limit_idling_pct', 'score_limit_late_minutes', 'score_limit_speeding_pct', 'score_weight_deviation', 'score_weight_idling', 'score_weight_lateness', 'score_weight_speeding', 'sequence_gap_grace_minutes', 'speed_max_kmh', 'speeding_tolerance_kmh', 'temperature_spike_c']


# Cell 6 The reusable functions

In [ ]:
def with_interval(df, expected_sec):
    """Each reading represents the time until the next reading, capped at CAP_MULT x expected interval.
       lost_sec = the part of a gap beyond the cap (nobody knows what happened)."""
    w = Window.partitionBy("trip_id").orderBy("event_time", "event_id")
    gap = (F.lead(F.col("event_time").cast("long")).over(w) - F.col("event_time").cast("long")).cast("double")
    cap = float(expected_sec * CAP_MULT)
    return (df.withColumn("_gap", gap)
              .withColumn("interval_sec", F.when(F.col("_gap").isNull(), F.lit(float(expected_sec)))
                                           .otherwise(F.least(F.col("_gap"), F.lit(cap))))
              .withColumn("lost_sec", F.when(F.col("_gap").isNull(), F.lit(0.0))
                                       .otherwise(F.greatest(F.col("_gap") - F.lit(cap), F.lit(0.0))))
              .drop("_gap"))

def build_episodes(df):
    """df columns: trip_id, event_id, event_time, interval_sec, lost_sec, is_bad(bool), value(double),
                   threshold(double), min_sec(double), min_readings(int).
       One output row per run of consecutive is_bad readings that lasts >= min_sec AND has >= min_readings readings."""
    w  = Window.partitionBy("trip_id").orderBy("event_time", "event_id")
    wc = w.rowsBetween(Window.unboundedPreceding, Window.currentRow)
    d = (df.withColumn("prev_bad",  F.lag("is_bad").over(w))
           .withColumn("prev_lost", F.lag("lost_sec").over(w))
           .withColumn("starts", F.when(F.col("is_bad") &
                        (F.coalesce(~F.col("prev_bad"), F.lit(True)) | (F.coalesce(F.col("prev_lost"), F.lit(0.0)) > 0)), 1).otherwise(0))
           .withColumn("run_no", F.sum("starts").over(wc)))
    runs = (d.filter("is_bad").groupBy("trip_id", "run_no").agg(
                F.min("event_time").alias("start_time"),
                F.sum("interval_sec").alias("duration_sec"),
                F.count("*").alias("readings"),
                F.max("value").alias("peak_value"),
                F.first("threshold").alias("threshold_value"),
                F.min_by("event_id", "event_time").alias("first_event_id"),
                F.max_by("event_id", "event_time").alias("last_event_id"),
                F.first("min_sec").alias("min_sec"), F.first("min_readings").alias("min_readings")))
    return (runs.filter((F.col("duration_sec") >= F.col("min_sec")) & (F.col("readings") >= F.col("min_readings")))
                .withColumn("end_time", F.timestamp_seconds(F.col("start_time").cast("long") + F.col("duration_sec")))
                .drop("min_sec", "min_readings", "run_no"))

# ---------- engine: idling ----------
def engine_prep(engine):
    e = with_interval(engine, EXP["engine"])
    on  = F.coalesce(F.col("engine_on"), F.lit(False))
    spd = F.col("vehicle_speed_kmh")
    return (e.withColumn("is_on", on)
             .withColumn("is_idle",    F.coalesce(on & (spd <  IDLE_KMH), F.lit(False)))
             .withColumn("is_driving", F.coalesce(on & (spd >= IDLE_KMH), F.lit(False))))

def idle_episodes(e):
    return build_episodes(e.select("trip_id", "event_id", "event_time", "interval_sec", "lost_sec",
                                   F.col("is_idle").alias("is_bad"), F.lit(None).cast("double").alias("value"),
                                   F.lit(IDLE_KMH).alias("threshold"),
                                   F.lit(IDLE_MIN * 60.0).alias("min_sec"), F.lit(1).alias("min_readings")))

def engine_measures(e, idle_ep):
    m = e.groupBy("trip_id").agg(
        (F.sum(F.when(F.col("is_on"),      F.col("interval_sec")).otherwise(0.0)) / 60).alias("engine_on_min"),
        (F.sum(F.when(F.col("is_driving"), F.col("interval_sec")).otherwise(0.0)) / 60).alias("driving_min"),
        (F.sum(F.when(F.col("is_idle"),    F.col("interval_sec")).otherwise(0.0)) / 60).alias("idle_min_total"),
        F.count("*").alias("engine_readings"))
    ex = idle_ep.groupBy("trip_id").agg((F.sum("duration_sec") / 60).alias("excessive_idle_min"))
    return m.join(ex, "trip_id", "left").fillna({"excessive_idle_min": 0.0})

# ---------- temperature: cold chain ----------
def temp_prep(temp, trips, cargo_controlled):
    """cargo_controlled: ref cargo rows with temperature limits only (dry goods excluded)."""
    t = temp_flags(temp)                                  # is_spike / is_flat, computed before the joins
    t = with_interval(t, EXP["temperature"])
    t = (t.join(trips.select("trip_id", "cargo_type_id"), "trip_id")
          .join(cargo_controlled.select("cargo_type_id", "temperature_min_c", "temperature_max_c", "alert_hold_minutes",
                                        "max_excursion_minutes", "cargo_value_per_unit", "spoilage_loss_pct", "currency_code"), "cargo_type_id"))
    sensor_ok = F.coalesce((F.col("sensor_status") == "OK") & F.col("temperature_c").isNotNull(), F.lit(False))
    valid = sensor_ok & ~F.col("is_spike") & ~F.col("is_flat")      # spikes and stuck sensors become UNMEASURED time
    inr = valid & (F.col("temperature_c") >= F.col("temperature_min_c")) & (F.col("temperature_c") <= F.col("temperature_max_c"))
    return (t.withColumn("is_valid", valid).withColumn("is_in_range", inr)
             .withColumn("is_out_range", valid & ~inr))

def breach_episodes(t):
    return build_episodes(
        t.select(
            "trip_id",
            "event_id",
            "event_time",
            "interval_sec",
            "lost_sec",
            F.col("is_out_range").alias("is_bad"),
            F.col("temperature_c").alias("value"),
            F.col("temperature_max_c").alias("threshold"),
            (F.col("alert_hold_minutes") * 60.0).alias("min_sec"),
            F.lit(1).alias("min_readings")
        )
    )

    
def temp_measures(t, breach_ep):
    m = t.groupBy("trip_id").agg(
        (F.sum(F.when(F.col("is_in_range"),  F.col("interval_sec")).otherwise(0.0)) / 60).alias("cold_in_range_min"),
        (F.sum(F.when(F.col("is_out_range"), F.col("interval_sec")).otherwise(0.0)) / 60).alias("cold_out_range_min"),
        ((F.sum(F.when(~F.col("is_valid"), F.col("interval_sec")).otherwise(0.0)) + F.sum("lost_sec")) / 60).alias("cold_unmeasured_min"),
        F.count("*").alias("temp_readings"),
        F.sum(F.when(F.col("is_spike") | F.col("is_flat"), 1).otherwise(0)).alias("suspect_readings"))
    m = m.withColumn("cold_compliance_pct",
            F.when(F.col("cold_in_range_min") + F.col("cold_out_range_min") > 0,
                   F.round(100 * F.col("cold_in_range_min") / (F.col("cold_in_range_min") + F.col("cold_out_range_min")), 2)))
    b = breach_ep.groupBy("trip_id").agg(F.count("*").alias("breach_episodes"))
    return m.join(b, "trip_id", "left").fillna({"breach_episodes": 0})

# ---------- fuel ----------
def fuel_measures(fuel):
    w = Window.partitionBy("trip_id").orderBy("event_time", "event_id")
    f = (fuel.withColumn("d_fuel", F.col("trip_fuel_used_liters") - F.lag("trip_fuel_used_liters").over(w))
             .withColumn("d_odo",  F.col("odometer_km") - F.lag("odometer_km").over(w)))
    m = f.groupBy("trip_id").agg(
        F.sum(F.when(F.col("d_odo")  > 0, F.col("d_odo")).otherwise(0.0)).alias("distance_km"),
        F.sum(F.when(F.col("d_fuel") > 0, F.col("d_fuel")).otherwise(0.0)).alias("fuel_used_l"),
        F.sum(F.when((F.col("d_fuel") < 0) | (F.col("d_odo") < 0), 1).otherwise(0)).alias("fuel_counter_anomalies"),
        F.count("*").alias("fuel_readings"))
    m = (m.withColumn("distance_km", F.when(F.col("fuel_readings") >= 2, F.col("distance_km")))     # 1 reading = unknown, not 0
           .withColumn("fuel_used_l", F.when(F.col("fuel_readings") >= 2, F.col("fuel_used_l"))))
    return m.withColumn("fuel_l_per_100km",
            F.when(F.col("distance_km") > 0, F.round(F.col("fuel_used_l") / F.col("distance_km") * 100, 2)))

# ---------- gps: deviation, speeding, arrival ----------
def dist_to_segment_m(lat, lon, lat1, lon1, lat2, lon2):
    k = F.lit(111320.0); cosl = F.cos(F.radians(lat1))
    px, py = (lon - lon1) * k * cosl, (lat - lat1) * k
    bx, by = (lon2 - lon1) * k * cosl, (lat2 - lat1) * k
    len2 = bx * bx + by * by
    t = F.greatest(F.lit(0.0), F.least(F.lit(1.0), (px * bx + py * by) / F.when(len2 == 0, F.lit(1.0)).otherwise(len2)))
    dx, dy = px - t * bx, py - t * by
    return F.sqrt(dx * dx + dy * dy)

def route_segments(checkpoints):
    w = Window.partitionBy("route_id").orderBy("sequence_number")
    return (checkpoints.withColumn("lat2", F.lead("latitude").over(w)).withColumn("lon2", F.lead("longitude").over(w))
            .filter("lat2 IS NOT NULL")
            .select("route_id", F.col("latitude").alias("lat1"), F.col("longitude").alias("lon1"), "lat2", "lon2"))

def route_destination(checkpoints):
    w = Window.partitionBy("route_id").orderBy(F.col("sequence_number").desc())
    return (checkpoints.withColumn("rn", F.row_number().over(w)).filter("rn = 1")
            .select("route_id", F.col("latitude").alias("dlat"), F.col("longitude").alias("dlon")))

def gps_prep(gps, trips, routes, seg, dest):
    gps = drop_gps_jumps(gps)
    g = gps.join(trips.select("trip_id", "route_id"), "trip_id")
    g = (g.join(F.broadcast(seg), "route_id")
          .withColumn("d_m", dist_to_segment_m(F.col("latitude"), F.col("longitude"),
                                               F.col("lat1"), F.col("lon1"), F.col("lat2"), F.col("lon2")))
          .groupBy("route_id", "trip_id", "event_id", "event_time", "latitude", "longitude", "speed_kmh", "gps_accuracy_meters")
          .agg(F.min("d_m").alias("dist_to_route_m")))
    g = with_interval(g, EXP["gps"])
    g = (g.join(routes.select("route_id", "speed_limit_kmh", "allowed_deviation_meters"), "route_id")
          .join(F.broadcast(dest), "route_id")
          .withColumn("dist_dest_m", F.sqrt(((F.col("latitude") - F.col("dlat")) * 111320.0) ** 2 +
                                            ((F.col("longitude") - F.col("dlon")) * 111320.0 * F.cos(F.radians("dlat"))) ** 2))
          .withColumn("is_speeding", F.col("speed_kmh") > F.col("speed_limit_kmh") + SPEED_TOL))
    return g

def deviation_episodes(g):
    ok = g.filter(F.col("gps_accuracy_meters") <= GPS_ACC_MAX)         # poor fixes are ignored, they do not break a run
    return build_episodes(ok.select("trip_id", "event_id", "event_time", "interval_sec", "lost_sec",
                                    (F.col("dist_to_route_m") > F.col("allowed_deviation_meters")).alias("is_bad"),
                                    F.col("dist_to_route_m").alias("value"),
                                    F.col("allowed_deviation_meters").cast("double").alias("threshold"),
                                    F.lit(0.0).alias("min_sec"), F.lit(DEV_READINGS).alias("min_readings")))

def gps_measures(g, dev_ep):
    m = g.groupBy("trip_id").agg(
        F.count("*").alias("gps_readings"),
        F.min("event_time").alias("actual_start_time"),
        F.min(F.when((F.col("dist_dest_m") <= ARRIVAL_RADIUS_M) & (F.col("gps_accuracy_meters") <= GPS_ACC_MAX),
                F.col("event_time"))).alias("actual_end_time"),
        (F.sum(F.when(F.col("is_speeding"), F.col("interval_sec")).otherwise(0.0)) / 60).alias("speeding_min"))
    dv = dev_ep.groupBy("trip_id").agg(F.count("*").alias("deviation_episodes"))
    return (m.join(dv, "trip_id", "left").fillna({"deviation_episodes": 0})
             .withColumn("arrived", F.col("actual_end_time").isNotNull()))

# ---------- near-duplicates, GPS teleports, temperature flags ----------
def drop_near_dups(df):
    """Same trip + same event_time with a different event_id = near-duplicate. First arrival wins (ties: event_id)."""
    w = Window.partitionBy("trip_id", "event_time").orderBy(F.col("ingestion_time").asc(), F.col("event_id").asc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

def haversine_km(lat1, lon1, lat2, lon2):
    a = (F.pow(F.sin(F.radians(lat2 - lat1) / 2), 2) +
         F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.pow(F.sin(F.radians(lon2 - lon1) / 2), 2))
    return 2 * 6371.0 * F.asin(F.sqrt(a))

def drop_gps_jumps(gps):
    """Remove isolated teleports: implied speed from BOTH neighbours above gps_jump_max_kmh. First/last rows are never dropped."""
    w = Window.partitionBy("trip_id").orderBy("event_time", "event_id")
    def implied_kmh(lat2, lon2, t2):
        dt_h = F.abs(F.col("event_time").cast("long") - t2.cast("long")) / 3600.0
        return F.when(dt_h > 0, haversine_km(F.col("latitude"), F.col("longitude"), lat2, lon2) / dt_h)
    prev_kmh = implied_kmh(F.lag("latitude").over(w), F.lag("longitude").over(w), F.lag("event_time").over(w))
    next_kmh = implied_kmh(F.lead("latitude").over(w), F.lead("longitude").over(w), F.lead("event_time").over(w))
    is_jump = F.coalesce((prev_kmh > JUMP_KMH) & (next_kmh > JUMP_KMH), F.lit(False))
    return gps.withColumn("_jump", is_jump).filter(~F.col("_jump")).drop("_jump")

def temp_flags(t):
    """is_spike: isolated point more than temperature_spike_c away from BOTH neighbours.
       is_flat : every member of a run of >= flatline_readings identical consecutive readings (stuck sensor)."""
    w = Window.partitionBy("trip_id").orderBy("event_time", "event_id")
    c = F.col("temperature_c")
    spike = (F.abs(c - F.lag(c).over(w)) > SPIKE_C) & (F.abs(c - F.lead(c).over(w)) > SPIKE_C)
    t = (t.withColumn("_chg", F.when(c.eqNullSafe(F.lag(c).over(w)), 0).otherwise(1))
          .withColumn("_run", F.sum("_chg").over(w.rowsBetween(Window.unboundedPreceding, Window.currentRow))))
    wr = Window.partitionBy("trip_id", "_run")
    return (t.withColumn("is_spike", F.coalesce(spike, F.lit(False)))
             .withColumn("is_flat", c.isNotNull() & (F.count("*").over(wr) >= FLAT_N))
             .drop("_chg", "_run"))

# ---------- trip arrival status and score ----------
def bad(num, lim_frac, den):
    return F.when(den > 0, F.least(F.lit(1.0), F.coalesce(num, F.lit(0.0)) / (F.lit(lim_frac) * den))).otherwise(F.lit(0.0))

def apply_arrival(df, fleet_now):
    """fleet_now = highest event_time seen in the fleet (epoch seconds). Never the wall clock."""
    deadline  = F.col("planned_end_time").cast("long") + F.coalesce(F.col("arrival_grace_minutes"), F.lit(0)) * 60
    reporting = (F.lit(fleet_now) - F.col("last_event_time").cast("long")) <= CAP_MULT * EXP["gps"]
    return (df.withColumn("arrived", F.coalesce(F.col("arrived"), F.lit(False)))
      .withColumn("has_telemetry", (F.col("gps_readings") + F.col("engine_readings")) > 0)
      .withColumn("arrival_status",
          F.when(F.col("trip_status") == "CANCELLED", "CANCELLED")
           .when(~F.col("has_telemetry"), "NO_DATA")
           .when(F.col("arrived"), "ARRIVED")
           .when((F.lit(fleet_now) <= deadline) | reporting, "IN_PROGRESS")
           .when(F.col("last_event_time") >= F.col("planned_end_time"), "NOT_ARRIVED")      # data covers the window, truck never arrived
           .otherwise("INCOMPLETE_TELEMETRY"))                                              # data stops early: unknown, not scored
      .withColumn("late_min", F.when(F.col("arrival_status") == "ARRIVED",
          F.greatest(F.lit(0.0), (F.col("actual_end_time").cast("long") - F.col("planned_end_time").cast("long")) / 60.0
                                  - F.coalesce(F.col("arrival_grace_minutes"), F.lit(0)))))
      .withColumn("score_complete", F.col("arrival_status").isin("ARRIVED", "NOT_ARRIVED")
                                    & (F.col("gps_readings") > 0) & (F.col("engine_readings") > 0)))

def apply_scores(df):
    complete = F.col("score_complete")
    def pen(e): return F.when(complete, e)
    late_badness = (F.when(F.col("arrival_status") == "NOT_ARRIVED", F.lit(1.0))
                     .otherwise(F.least(F.lit(1.0), F.coalesce(F.col("late_min"), F.lit(0.0)) / F.lit(LIM_LT))))
    return (df
      .withColumn("score_speeding_penalty",  pen(F.lit(W_SP) * bad(F.col("speeding_min"), LIM_SP, F.col("driving_min"))))
      .withColumn("score_idling_penalty",    pen(F.lit(W_ID) * bad(F.col("excessive_idle_min"), LIM_ID, F.col("engine_on_min"))))
      .withColumn("score_deviation_penalty", pen(F.lit(W_DV) * F.least(F.lit(1.0), F.coalesce(F.col("deviation_episodes"), F.lit(0)) / F.lit(LIM_DV))))
      .withColumn("score_lateness_penalty",  pen(F.lit(W_LT) * late_badness))
      .withColumn("trip_score", F.when(complete, F.round(100 - F.col("score_speeding_penalty") - F.col("score_idling_penalty")
                                                         - F.col("score_deviation_penalty") - F.col("score_lateness_penalty"), 1))))


StatementMeta(, a28a30e1-c1ac-4b6d-b217-1c2a6a75becb, 41, Finished, Available, Finished, False)

# Cell 6 validation

In [ ]:
from datetime import datetime
fx = spark.createDataFrame([
    ("a", "T1", datetime(2026, 1, 1, 0, 0, 0)),
    ("b", "T1", datetime(2026, 1, 1, 0, 0, 10)),
    ("c", "T1", datetime(2026, 1, 1, 0, 1, 0)),
], ["event_id", "trip_id", "event_time"])
wi = with_interval(fx, 10).orderBy("event_time")
rows = wi.select("interval_sec", "lost_sec").collect()
assert rows[0]["interval_sec"] == 10 and rows[0]["lost_sec"] == 0, rows
assert rows[1]["interval_sec"] == 30 and rows[1]["lost_sec"] == 20, rows
assert rows[2]["interval_sec"] == 10 and rows[2]["lost_sec"] == 0, rows
print("Cell 6 interval helper passed")

StatementMeta(, a28a30e1-c1ac-4b6d-b217-1c2a6a75becb, 42, Finished, Available, Finished, False)

Cell 6 interval helper passed


In [ ]:
from datetime import datetime, timedelta
from pyspark.sql import Row
t0 = datetime(2026, 1, 1, 6, 0, 0)
def ts(sec): return t0 + timedelta(seconds=sec)

# 1) build_episodes: pattern 0,1,1,1,0,0,1,0 at 30 s, min 60 s -> ONE episode of 90 s (the lone bad reading is too short)
rows = [(f"e{i}", "T1", ts(30*i), 30.0, 0.0, bool(b), 9.0, 8.0, 60.0, 1) for i, b in enumerate([0,1,1,1,0,0,1,0])]
df = spark.createDataFrame(rows, "event_id string, trip_id string, event_time timestamp, interval_sec double, lost_sec double, is_bad boolean, value double, threshold double, min_sec double, min_readings int")
ep = build_episodes(df).collect()
assert len(ep) == 1 and ep[0]["duration_sec"] == 90.0 and ep[0]["readings"] == 3, ep

# 2) temperature: FX1 breach, FX2 isolated spike, FX3 stuck sensor
fx_trips = spark.createDataFrame([Row(trip_id=x, route_id="RX", cargo_type_id="FOOD_CHILLED") for x in ("FX1", "FX2", "FX3")])
fx_cargo = spark.createDataFrame([Row(cargo_type_id="FOOD_CHILLED", temperature_min_c=2.0, temperature_max_c=8.0,
    alert_hold_minutes=5, max_excursion_minutes=10, cargo_value_per_unit=100.0, spoilage_loss_pct=0.25, currency_code="USD")])
def temp_rows(trip, vals):
    return [Row(event_id=f"{trip}-{i}", trip_id=trip, vehicle_id="VX", event_time=ts(30*i),
                temperature_c=float(v), sensor_status="OK") for i, v in enumerate(vals)]
fx_temp = spark.createDataFrame(temp_rows("FX1", [4.0]*4 + [9.0]*10 + [4.0]*6)
                              + temp_rows("FX2", [4.0]*5 + [20.0] + [4.0]*5)
                              + temp_rows("FX3", [4.0]*(FLAT_N + 5)))
tp = temp_prep(fx_temp, fx_trips, fx_cargo)
tm = {r["trip_id"]: r for r in temp_measures(tp, breach_episodes(tp)).collect()}
a, b, c = tm["FX1"], tm["FX2"], tm["FX3"]
assert abs(a.cold_in_range_min - 5.0) < 1e-6 and abs(a.cold_out_range_min - 5.0) < 1e-6 and a.cold_compliance_pct == 50.0 and a.breach_episodes == 1, a
assert abs(b.cold_unmeasured_min - 0.5) < 1e-6 and abs(b.cold_in_range_min - 5.0) < 1e-6 and b.suspect_readings == 1 and b.breach_episodes == 0, b
assert abs(c.cold_in_range_min) < 1e-6 and abs(c.cold_unmeasured_min - (FLAT_N + 5) * 0.5) < 1e-6, c

# 3) engine: 3-minute stop is normal, 6-minute stop is excessive idling
speeds = [60]*6 + [0]*18 + [60]*6 + [0]*36 + [60]*18
fx_eng = spark.createDataFrame([Row(event_id=f"e{i}", trip_id="FX1", vehicle_id="VX", event_time=ts(10*i),
                                    engine_on=True, vehicle_speed_kmh=float(s)) for i, s in enumerate(speeds)])
ep_ = engine_prep(fx_eng); em = engine_measures(ep_, idle_episodes(ep_)).first()
assert abs(em.engine_on_min - 14.0) < 1e-6 and abs(em.driving_min - 5.0) < 1e-6, em
assert abs(em.idle_min_total - 9.0) < 1e-6 and abs(em.excessive_idle_min - 6.0) < 1e-6, em

# 4) fuel: normal trip, and a trip whose counters reset
def fuel_rows(trip, pairs):
    return [Row(event_id=f"{trip}-{i}", trip_id=trip, event_time=ts(60*i), odometer_km=float(o), trip_fuel_used_liters=float(l))
            for i, (o, l) in enumerate(pairs)]
fx_fuel = spark.createDataFrame(fuel_rows("FX1", [(1000,0),(1025,5),(1050,10),(1100,20)])
                              + fuel_rows("FX5", [(1000,0),(1025,5),(1050,10),(0,0),(25,5),(50,10)]))
fm = {r["trip_id"]: r for r in fuel_measures(fx_fuel).collect()}
assert fm["FX1"].distance_km == 100.0 and fm["FX1"].fuel_used_l == 20.0 and fm["FX1"].fuel_l_per_100km == 20.0 and fm["FX1"].fuel_counter_anomalies == 0, fm["FX1"]
assert fm["FX5"].distance_km == 100.0 and fm["FX5"].fuel_used_l == 20.0 and fm["FX5"].fuel_counter_anomalies == 1, fm["FX5"]

# 5) GPS teleport removed, normal points kept
pts = [(f"g{i}", "FX4", ts(10*i), 30.0 + 0.0005*i, 31.0) for i in range(8)]
pts[3] = ("g3", "FX4", ts(30), 35.0, 31.0)
fx_gps = spark.createDataFrame(pts, "event_id string, trip_id string, event_time timestamp, latitude double, longitude double")
kept = {r["event_id"] for r in drop_gps_jumps(fx_gps).collect()}
assert kept == {f"g{i}" for i in range(8)} - {"g3"}, kept

# 6) arrival status (fleet clock = 18:00)
d = lambda h, m=0: datetime(2026, 1, 1, h, m, 0)
arr = [("A","SCHEDULED",10,10,True, d(12),15,d(12,30),d(12,20)), ("B","SCHEDULED",10,10,False,d(12),15,d(12,30),None),
       ("C","SCHEDULED",10,10,False,d(12),15,d(11),None),        ("D","SCHEDULED",10,10,False,d(17,50),15,d(17),None),
       ("E","SCHEDULED",0,0,False,d(12),15,None,None),           ("F","CANCELLED",0,0,False,d(12),15,None,None)]
fx_arr = spark.createDataFrame(arr, "trip_id string, trip_status string, gps_readings long, engine_readings long, arrived boolean, planned_end_time timestamp, arrival_grace_minutes int, last_event_time timestamp, actual_end_time timestamp")
fleet_now = fx_arr.agg(F.min(F.col("planned_end_time").cast("long"))).first()[0] + 6*3600
res = {r["trip_id"]: r for r in apply_arrival(fx_arr, fleet_now).collect()}
assert {k: v.arrival_status for k, v in res.items()} == {"A":"ARRIVED","B":"NOT_ARRIVED","C":"INCOMPLETE_TELEMETRY","D":"IN_PROGRESS","E":"NO_DATA","F":"CANCELLED"}, res
assert abs(res["A"].late_min - 5.0) < 1e-6 and res["A"].score_complete and res["B"].score_complete and not res["C"].score_complete

# 7) score: the guide's worked example (80.7), a never-arrived trip (68.2), and an unscored trip (NULL)
sc = spark.createDataFrame([("S1", True, "ARRIVED", 4.0, 360.0, 12.0, 400.0, 0, 10.0),
                            ("S2", True, "NOT_ARRIVED", 4.0, 360.0, 12.0, 400.0, 0, None),
                            ("S3", False, "IN_PROGRESS", 4.0, 360.0, 12.0, 400.0, 0, None)],
  "trip_id string, score_complete boolean, arrival_status string, speeding_min double, driving_min double, excessive_idle_min double, engine_on_min double, deviation_episodes long, late_min double")
r = {x["trip_id"]: x for x in apply_scores(sc).collect()}
assert abs(r["S1"].trip_score - 80.7) < 0.05 and abs(r["S2"].trip_score - 68.2) < 0.05 and r["S3"].trip_score is None, r
print("fixtures OK")

StatementMeta(, a28a30e1-c1ac-4b6d-b217-1c2a6a75becb, 43, Finished, Available, Finished, False)

fixtures OK


### Cell 7 — read inputs and build fact_trip_performance; fact_cold_chain is a separate required transformation

In [ ]:
gps    = drop_near_dups(spark.table("gold.v_in_gps"))
engine = drop_near_dups(spark.table("gold.v_in_engine"))
fuel   = drop_near_dups(spark.table("gold.v_in_fuel"))
temp   = drop_near_dups(spark.table("gold.v_in_temperature"))

trips   = spark.table("ref.ref_trips")
routes  = spark.table("ref.ref_routes")
cps     = spark.table("ref.ref_route_checkpoints")
cargo   = spark.table("ref.ref_cargo_types")
cargo_c = cargo.filter("temperature_max_c IS NOT NULL")
dv      = spark.table("gold.dim_vehicle")

for name, df in {"gps":gps, "engine":engine, "fuel":fuel, "temperature":temp, "trips":trips,
                 "routes":routes, "checkpoints":cps, "cargo":cargo, "dim_vehicle":dv}.items():
    assert df.limit(1).count() == 1, f"Required input {name} is empty"
assert trips.count() == trips.select("trip_id").distinct().count(), "ref.ref_trips has duplicate trip_id"

tk = (trips.alias("t")
      .join(dv.alias("v"), (F.col("t.vehicle_id") == F.col("v.vehicle_id")) &
                           (F.col("t.planned_start_time") >= F.col("v.effective_from")) &
                           (F.col("t.planned_start_time") <  F.col("v.effective_to")), "left")
      .join(routes.alias("r"), F.col("t.route_id") == F.col("r.route_id"), "left")
      .join(cargo.alias("c"), F.col("t.cargo_type_id") == F.col("c.cargo_type_id"), "left")
      .select("t.*",
              F.coalesce(F.col("v.vehicle_key"), F.lit(-1)).alias("vehicle_key"),
              F.col("v.refrigeration_capable").alias("veh_refrigerated"),
              F.col("r.arrival_grace_minutes").alias("arrival_grace_minutes"),
              F.col("c.temperature_max_c").isNotNull().alias("cargo_controlled"))
      .withColumn("driver_key", F.xxhash64("driver_id"))
      .withColumn("route_key", F.xxhash64("route_id"))
      .withColumn("cargo_type_key", F.xxhash64("cargo_type_id"))
      .withColumn("planned_start_date_key", F.date_format("planned_start_time", "yyyyMMdd").cast("int")))

# each signal aggregated to trip grain on its own, never joined raw
e = engine_prep(engine); idle_ep = idle_episodes(e); eng_m = engine_measures(e, idle_ep)
t = temp_prep(temp, trips, cargo_c); br_ep = breach_episodes(t); tmp_m = temp_measures(t, br_ep)
exc_ep = build_episodes(t.select("trip_id", "event_id", "event_time", "interval_sec", "lost_sec",
    F.col("is_out_range").alias("is_bad"), F.col("temperature_c").alias("value"),
    F.col("temperature_max_c").cast("double").alias("threshold"), F.lit(0.0).alias("min_sec"), F.lit(1).alias("min_readings")))
exc_m = exc_ep.groupBy("trip_id").agg(F.count("*").alias("excursion_episodes"),
                                      (F.max("duration_sec") / 60.0).alias("longest_excursion_min"))
g = gps_prep(gps, trips, routes, route_segments(cps), route_destination(cps))
dev_ep = deviation_episodes(g); gps_m = gps_measures(g, dev_ep)
fuel_m = fuel_measures(fuel)

ev_times = (gps.select("trip_id", "event_time").unionByName(engine.select("trip_id", "event_time"))
               .unionByName(fuel.select("trip_id", "event_time")).unionByName(temp.select("trip_id", "event_time"))
               .groupBy("trip_id").agg(F.min("event_time").alias("first_event_time"), F.max("event_time").alias("last_event_time")))
FLEET_NOW = ev_times.agg(F.max(F.col("last_event_time").cast("long"))).first()[0]      # fleet stream time, epoch seconds

# FACT 1: only COUNTS are zero-filled; missing measures stay NULL (unknown is not zero)
f = (tk.join(fuel_m, "trip_id", "left").join(eng_m, "trip_id", "left").join(gps_m, "trip_id", "left")
       .join(ev_times, "trip_id", "left")
       .fillna(0, subset=["gps_readings", "engine_readings", "fuel_readings", "fuel_counter_anomalies"]))
f = apply_scores(apply_arrival(f, FLEET_NOW))
f = (f.withColumn("invalid_assignment", F.col("cargo_controlled") & ~F.coalesce(F.col("veh_refrigerated"), F.lit(False)))
      .withColumn("built_at", F.current_timestamp()))

FACT_TRIP_COLS = ["trip_id","vehicle_key","driver_key","route_key","cargo_type_key","planned_start_date_key",
  "vehicle_id","driver_id","route_id","shipment_id","cargo_type_id","planned_start_time","planned_end_time","cargo_quantity","trip_status",
  "actual_start_time","actual_end_time","arrived","arrival_status","late_min",
  "distance_km","fuel_used_l","fuel_l_per_100km","fuel_counter_anomalies",
  "engine_on_min","driving_min","idle_min_total","excessive_idle_min","speeding_min","deviation_episodes",
  "score_speeding_penalty","score_idling_penalty","score_deviation_penalty","score_lateness_penalty","trip_score","score_complete",
  "invalid_assignment","has_telemetry","gps_readings","engine_readings","fuel_readings",
  "first_event_time","last_event_time","built_at"]
f.select(*FACT_TRIP_COLS).write.format("delta").mode("overwrite").saveAsTable("gold.fact_trip_performance")

# FACT 2: eligible trips = cold cargo AND a refrigerated vehicle version at planned start
cc_base = (tk.filter(F.col("cargo_controlled") & F.coalesce(F.col("veh_refrigerated"), F.lit(False)))
    .select("trip_id", "shipment_id", "vehicle_key", "driver_key", "route_key", "cargo_type_key", "planned_start_date_key",
            "vehicle_id", "driver_id", "route_id", "cargo_type_id", "cargo_quantity"))
cc = (cc_base.join(tmp_m, "trip_id", "left").join(exc_m, "trip_id", "left")
    .join(cargo.select("cargo_type_id","cargo_value_per_unit",(F.col("spoilage_loss_pct") / 100.0).alias("spoilage_loss_pct"),"max_excursion_minutes"), "cargo_type_id", "left")
    .withColumn("in_range_min",   F.coalesce(F.col("cold_in_range_min"), F.lit(0.0)))
    .withColumn("out_range_min",  F.coalesce(F.col("cold_out_range_min"), F.lit(0.0)))
    .withColumn("unmeasured_min", F.coalesce(F.col("cold_unmeasured_min"), F.lit(0.0)))
    .withColumn("measured_min",   F.col("in_range_min") + F.col("out_range_min"))
    .withColumn("breach_episodes",    F.coalesce(F.col("breach_episodes"), F.lit(0)))
    .withColumn("excursion_episodes", F.coalesce(F.col("excursion_episodes"), F.lit(0)))
    .withColumn("suspect_readings",   F.coalesce(F.col("suspect_readings"), F.lit(0)))
    .withColumn("breach_flag", F.col("breach_episodes") > 0)
    .withColumn("longest_excursion_min", F.coalesce(F.col("longest_excursion_min"), F.lit(0.0)))
    .withColumn("compromised_flag", F.col("longest_excursion_min") > F.coalesce(F.col("max_excursion_minutes"), F.lit(float("inf"))))
    .join(t.groupBy("trip_id").agg(F.min(F.when(F.col("is_valid"), F.col("temperature_c"))).alias("min_temperature_c"),
                                    F.max(F.when(F.col("is_valid"), F.col("temperature_c"))).alias("max_temperature_c")), "trip_id", "left")
    .withColumn("cargo_value", F.col("cargo_quantity") * F.col("cargo_value_per_unit"))
    .withColumn("cargo_at_risk", F.when(F.col("compromised_flag"), F.col("cargo_value") * F.col("spoilage_loss_pct")).otherwise(F.lit(0.0)))
    .withColumn("temp_readings", F.coalesce(F.col("temp_readings"), F.lit(0)))
    .withColumn("built_at", F.current_timestamp()))
FACT_CC_COLS = ["trip_id","shipment_id","vehicle_key","driver_key","route_key","cargo_type_key","planned_start_date_key",
  "vehicle_id","driver_id","route_id","cargo_type_id","measured_min","in_range_min","out_range_min","unmeasured_min",
  "min_temperature_c","max_temperature_c","excursion_episodes","breach_episodes","longest_excursion_min",
  "breach_flag","compromised_flag","cargo_value","cargo_at_risk","suspect_readings","temp_readings","built_at"]
cc.select(*FACT_CC_COLS).write.format("delta").mode("overwrite").saveAsTable("gold.fact_cold_chain")

print("fact_trip_performance rows:", spark.table("gold.fact_trip_performance").count())
print("fact_cold_chain rows:", spark.table("gold.fact_cold_chain").count())

StatementMeta(, a28a30e1-c1ac-4b6d-b217-1c2a6a75becb, 57, Finished, Available, Finished, False)

fact_trip_performance rows: 21


fact_cold_chain rows: 14


# Cell 7 validation

In [ ]:
trip = spark.table("gold.fact_trip_performance")
cc   = spark.table("gold.fact_cold_chain")
expected_trips = spark.table("ref.ref_trips").count()
assert trip.count() == expected_trips, f"Expected one trip row per ref trip: {trip.count()} vs {expected_trips}"
assert trip.count() == trip.select("trip_id").distinct().count(), "Duplicate trip_id in fact_trip_performance"
assert cc.count() == cc.select("trip_id").distinct().count(), "Duplicate trip_id in fact_cold_chain"
assert cc.join(trip.select("trip_id"), "trip_id", "left_anti").count() == 0, "Cold-chain trip missing from trip fact"
assert trip.filter("vehicle_key = -1").count() == 0, "Trip without a vehicle version at planned start (vehicle_key = -1)"
assert trip.filter("distance_km < 0 OR fuel_used_l < 0 OR driving_min < 0 OR excessive_idle_min < 0").count() == 0, "Negative measures"
assert cc.filter("measured_min < 0 OR in_range_min < 0 OR out_range_min < 0 OR unmeasured_min < 0 OR cargo_at_risk < 0").count() == 0, "Negative cold-chain measures"
assert cc.filter("cargo_at_risk > cargo_value + 0.01").count() == 0, "cargo_at_risk > cargo_value: spoilage_loss_pct is probably a percent, not a fraction"
assert cc.filter("breach_episodes > excursion_episodes").count() == 0, "a qualifying breach must also be an excursion run"
assert trip.filter("score_complete AND trip_score IS NULL").count() == 0, "complete trip without a score"
assert trip.filter("NOT score_complete AND trip_score IS NOT NULL").count() == 0, "incomplete trip must not be scored"
assert trip.filter("trip_score < 0 OR trip_score > 100").count() == 0, "score out of range"

# time reconciliation: measured + unmeasured must equal the trip's temperature span (+ one interval)
span = (drop_near_dups(spark.table("gold.v_in_temperature")).groupBy("trip_id")
        .agg(((F.max(F.col("event_time").cast("long")) - F.min(F.col("event_time").cast("long"))) / 60.0 + EXP["temperature"] / 60.0).alias("span_min")))
assert cc.join(span, "trip_id").filter(F.abs(F.col("measured_min") + F.col("unmeasured_min") - F.col("span_min")) > 0.01).count() == 0, \
    "measured + unmeasured does not match the temperature time span"

trip.groupBy("arrival_status").count().show()
print("fuel counter anomalies:", trip.agg(F.sum("fuel_counter_anomalies")).first()[0],
      "| suspect temperature readings:", cc.agg(F.sum("suspect_readings")).first()[0])
print("Cell 7 passed")

StatementMeta(, a28a30e1-c1ac-4b6d-b217-1c2a6a75becb, 61, Finished, Available, Finished, False)

+--------------+-----+
|arrival_status|count|
+--------------+-----+
|       ARRIVED|   21|
+--------------+-----+



fuel counter anomalies: 0 | suspect temperature readings: 44
Cell 7 passed


# Dashboard tables

### Cell 1

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold.agg_route_kpi_daily USING DELTA AS
WITH trip_daily AS (
  SELECT f.route_id, r.route_name, d.date AS trip_date,
         COUNT(*) AS trips,
         COUNT(f.distance_km) AS trips_with_fuel_data,
         SUM(f.distance_km) AS distance_km,
         SUM(f.fuel_used_l) AS fuel_used_l,
         CASE WHEN SUM(f.distance_km) > 0 THEN ROUND(SUM(f.fuel_used_l) / SUM(f.distance_km) * 100, 2) END AS fuel_l_per_100km
  FROM gold.fact_trip_performance f
  JOIN gold.dim_route r ON f.route_key = r.route_key
  JOIN gold.dim_date d ON f.planned_start_date_key = d.date_key
  WHERE f.has_telemetry
  GROUP BY f.route_id, r.route_name, d.date
), cold_daily AS (
  SELECT f.route_id, d.date AS trip_date,
         COUNT(*) AS cold_trips,
         SUM(cc.in_range_min) AS cold_in_range_min,
         SUM(cc.out_range_min) AS cold_out_range_min,
         SUM(cc.unmeasured_min) AS cold_unmeasured_min,
         CASE WHEN SUM(cc.in_range_min) + SUM(cc.out_range_min) > 0
              THEN ROUND(100 * SUM(cc.in_range_min) / (SUM(cc.in_range_min) + SUM(cc.out_range_min)), 2) END AS cold_compliance_pct,
         SUM(cc.breach_episodes) AS breach_episodes,
         SUM(cc.excursion_episodes) AS excursion_episodes,
         SUM(CASE WHEN cc.breach_flag THEN 1 ELSE 0 END) AS breach_trips,
         SUM(CASE WHEN cc.compromised_flag THEN 1 ELSE 0 END) AS compromised_trips,
         SUM(cc.cargo_value) AS cargo_value,
         SUM(cc.cargo_at_risk) AS cargo_at_risk
  FROM gold.fact_cold_chain cc
  JOIN gold.fact_trip_performance f ON cc.trip_id = f.trip_id
  JOIN gold.dim_date d ON f.planned_start_date_key = d.date_key
  WHERE f.has_telemetry
  GROUP BY f.route_id, d.date
)
SELECT t.route_id, t.route_name, t.trip_date, t.trips, t.trips_with_fuel_data, t.distance_km, t.fuel_used_l, t.fuel_l_per_100km,
       c.cold_trips, c.cold_in_range_min, c.cold_out_range_min, c.cold_unmeasured_min, c.cold_compliance_pct,
       c.breach_episodes, c.excursion_episodes, c.breach_trips, c.compromised_trips, c.cargo_value, c.cargo_at_risk
FROM trip_daily t LEFT JOIN cold_daily c ON t.route_id = c.route_id AND t.trip_date = c.trip_date;

CREATE OR REPLACE TABLE gold.agg_driver_scorecard_daily USING DELTA AS
SELECT x.*,
       CASE WHEN driver_score >= 85 THEN 'Good' WHEN driver_score >= 70 THEN 'Watch' ELSE 'At risk' END AS score_band
FROM (
  SELECT f.driver_id, dr.driver_name, d.date AS trip_date,
         COUNT(*) AS trips,
         SUM(f.driving_min)        AS driving_min,
         SUM(f.excessive_idle_min) AS excessive_idle_min,
         SUM(f.speeding_min)       AS speeding_min,
         SUM(f.deviation_episodes) AS deviation_episodes,
         SUM(CASE WHEN f.late_min > 0 THEN 1 ELSE 0 END) AS late_trips,
         SUM(CASE WHEN f.arrival_status = 'NOT_ARRIVED' THEN 1 ELSE 0 END) AS not_arrived_trips,
         SUM(f.trip_score * f.driving_min) AS score_x_min,
         ROUND(CASE WHEN SUM(f.driving_min) > 0 THEN SUM(f.trip_score * f.driving_min) / SUM(f.driving_min)
                    ELSE AVG(f.trip_score) END, 1) AS driver_score
  FROM gold.fact_trip_performance f
  JOIN gold.dim_driver dr ON f.driver_key = dr.driver_key
  JOIN gold.dim_date   d  ON f.planned_start_date_key = d.date_key
  WHERE f.score_complete
  GROUP BY f.driver_id, dr.driver_name, d.date
) x;

CREATE OR REPLACE VIEW gold.v_driver_scorecard_monthly AS
SELECT m.*,
       CASE WHEN driver_score >= 85 THEN 'Good' WHEN driver_score >= 70 THEN 'Watch' ELSE 'At risk' END AS score_band
FROM (
  SELECT driver_id, driver_name, date_format(trip_date, 'yyyy-MM') AS month,
         SUM(trips) AS trips, SUM(driving_min) AS driving_min, SUM(excessive_idle_min) AS excessive_idle_min,
         SUM(speeding_min) AS speeding_min, SUM(deviation_episodes) AS deviation_episodes,
         SUM(late_trips) AS late_trips, SUM(not_arrived_trips) AS not_arrived_trips,
         ROUND(CASE WHEN SUM(driving_min) > 0 THEN SUM(score_x_min) / SUM(driving_min) ELSE AVG(driver_score) END, 1) AS driver_score
  FROM gold.agg_driver_scorecard_daily
  GROUP BY driver_id, driver_name, date_format(trip_date, 'yyyy-MM')
) m;

StatementMeta(, a28a30e1-c1ac-4b6d-b217-1c2a6a75becb, 67, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

## Cell 1 validation

In [ ]:
%%sql
SELECT route_id, trip_date, COUNT(*) n FROM gold.agg_route_kpi_daily GROUP BY route_id, trip_date HAVING COUNT(*) <> 1;
SELECT driver_id, trip_date, COUNT(*) n FROM gold.agg_driver_scorecard_daily GROUP BY driver_id, trip_date HAVING COUNT(*) <> 1;
SELECT COUNT(*) AS negative_route_measures FROM gold.agg_route_kpi_daily WHERE distance_km < 0 OR fuel_used_l < 0;
SELECT COUNT(*) AS bad_scores FROM gold.agg_driver_scorecard_daily WHERE driver_score < 0 OR driver_score > 100;
SELECT (SELECT SUM(trips) FROM gold.agg_route_kpi_daily) - (SELECT COUNT(*) FROM gold.fact_trip_performance WHERE has_telemetry) AS route_trips_diff;
SELECT (SELECT SUM(trips) FROM gold.agg_driver_scorecard_daily) - (SELECT COUNT(*) FROM gold.fact_trip_performance WHERE score_complete) AS driver_trips_diff;
SELECT ROUND((SELECT COALESCE(SUM(cold_in_range_min),0) FROM gold.agg_route_kpi_daily)
           - (SELECT COALESCE(SUM(cc.in_range_min),0) FROM gold.fact_cold_chain cc JOIN gold.fact_trip_performance f ON cc.trip_id = f.trip_id WHERE f.has_telemetry), 4) AS cold_in_range_diff;

StatementMeta(, a28a30e1-c1ac-4b6d-b217-1c2a6a75becb, 74, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 3 fields>

<Spark SQL result set with 0 rows and 3 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

## Tests (the first successful build)

In [ ]:
%%sql
-- duplicate keys: both must return 0
SELECT COUNT(*) FROM (SELECT trip_id FROM gold.fact_trip_performance GROUP BY trip_id HAVING COUNT(*) > 1);
SELECT COUNT(*) FROM (SELECT trip_id FROM gold.fact_cold_chain GROUP BY trip_id HAVING COUNT(*) > 1);
SELECT COUNT(*) FROM (SELECT vehicle_id FROM gold.dim_vehicle WHERE is_current GROUP BY vehicle_id HAVING COUNT(*) <> 1);
-- every fact key finds its dimension (each must return 0)
SELECT COUNT(*) FROM gold.fact_trip_performance f LEFT JOIN gold.dim_vehicle d ON f.vehicle_key=d.vehicle_key WHERE d.vehicle_key IS NULL;
SELECT COUNT(*) FROM gold.fact_trip_performance f LEFT JOIN gold.dim_driver d ON f.driver_key=d.driver_key WHERE d.driver_key IS NULL;
SELECT COUNT(*) FROM gold.fact_trip_performance f LEFT JOIN gold.dim_route d ON f.route_key=d.route_key WHERE d.route_key IS NULL;
SELECT COUNT(*) FROM gold.fact_trip_performance f LEFT JOIN gold.dim_cargo_type d ON f.cargo_type_key=d.cargo_type_key WHERE d.cargo_type_key IS NULL;
SELECT COUNT(*) FROM gold.fact_cold_chain f LEFT JOIN gold.dim_vehicle d ON f.vehicle_key=d.vehicle_key WHERE d.vehicle_key IS NULL;
SELECT COUNT(*) FROM gold.fact_cold_chain f LEFT JOIN gold.dim_driver d ON f.driver_key=d.driver_key WHERE f.driver_key IS NOT NULL AND d.driver_key IS NULL;
SELECT COUNT(*) FROM gold.fact_cold_chain f LEFT JOIN gold.dim_route d ON f.route_key=d.route_key WHERE d.route_key IS NULL;
SELECT COUNT(*) FROM gold.fact_cold_chain f LEFT JOIN gold.dim_cargo_type d ON f.cargo_type_key=d.cargo_type_key WHERE d.cargo_type_key IS NULL;
-- non-overlapping vehicle versions
SELECT COUNT(*) FROM gold.dim_vehicle a JOIN gold.dim_vehicle b
  ON a.vehicle_id = b.vehicle_id AND a.vehicle_key <> b.vehicle_key
 AND a.effective_from < b.effective_to AND b.effective_from < a.effective_to;
-- one row per trip in ref_trips
SELECT (SELECT COUNT(*) FROM ref.ref_trips) - (SELECT COUNT(*) FROM gold.fact_trip_performance);
-- invalid cold-cargo assignments (informational: report them, do not hide them)
SELECT trip_id, vehicle_id, cargo_type_id, planned_start_time FROM gold.fact_trip_performance WHERE invalid_assignment;

StatementMeta(, 0aba5391-1e24-4e3a-9cef-b32e7aeda0b6, 48, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

...

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 0 rows and 4 fields>

### SCD Test

In [ ]:
%%sql
-- vehicles that have history
SELECT vehicle_id, vehicle_key, vehicle_type, refrigeration_capable, device_id, registration_number,
       effective_from, effective_to, is_current
FROM gold.dim_vehicle WHERE vehicle_id IN (SELECT vehicle_id FROM gold.dim_vehicle GROUP BY vehicle_id HAVING COUNT(*) > 1)
ORDER BY vehicle_id, effective_from;

-- a trip before and a trip after the change must carry different vehicle_key values
SELECT f.trip_id, f.vehicle_id, f.planned_start_time, f.vehicle_key
FROM gold.fact_trip_performance f WHERE f.vehicle_id = 'V002' ORDER BY f.planned_start_time;

StatementMeta(, 680f11e0-0c06-4ff5-bda1-9d303d1dc676, 5, Finished, Available, Finished, True)

<Spark SQL result set with 4 rows and 9 fields>

<Spark SQL result set with 2 rows and 4 fields>

### Gate A — reference data (all must return 0 unless stated)

In [1]:
%%sql
-- A1 row counts (expect: all > 0)
SELECT 'vehicles' t, COUNT(*) n FROM ref.ref_vehicles_raw UNION ALL SELECT 'trips', COUNT(*) FROM ref.ref_trips
UNION ALL SELECT 'drivers', COUNT(*) FROM ref.ref_drivers UNION ALL SELECT 'routes', COUNT(*) FROM ref.ref_routes
UNION ALL SELECT 'checkpoints', COUNT(*) FROM ref.ref_route_checkpoints UNION ALL SELECT 'cargo', COUNT(*) FROM ref.ref_cargo_types;

-- A2 every trip points at something that exists (each must be 0)
SELECT COUNT(*) AS orphan_vehicle FROM ref.ref_trips t LEFT JOIN (SELECT DISTINCT vehicle_id FROM ref.ref_vehicles_raw) v ON t.vehicle_id = v.vehicle_id WHERE v.vehicle_id IS NULL;
SELECT COUNT(*) AS orphan_driver  FROM ref.ref_trips t LEFT JOIN ref.ref_drivers d       ON t.driver_id = d.driver_id           WHERE d.driver_id IS NULL;
SELECT COUNT(*) AS orphan_route   FROM ref.ref_trips t LEFT JOIN ref.ref_routes r        ON t.route_id = r.route_id             WHERE r.route_id IS NULL;
SELECT COUNT(*) AS orphan_cargo   FROM ref.ref_trips t LEFT JOIN ref.ref_cargo_types c   ON t.cargo_type_id = c.cargo_type_id   WHERE c.cargo_type_id IS NULL;   -- DRY_GOODS missing shows up here

-- A3 trip ids unique
SELECT COUNT(*) - COUNT(DISTINCT trip_id) AS duplicate_trips FROM ref.ref_trips;

-- A4 checkpoints: every route has >= 2 points numbered 1..n without holes (expect 0 rows)
SELECT route_id, COUNT(*) n, MIN(sequence_number) mn, MAX(sequence_number) mx
FROM ref.ref_route_checkpoints GROUP BY route_id HAVING COUNT(*) < 2 OR MIN(sequence_number) <> 1 OR MAX(sequence_number) <> COUNT(*);
SELECT COUNT(*) AS routes_without_checkpoints FROM ref.ref_routes r LEFT JOIN (SELECT DISTINCT route_id FROM ref.ref_route_checkpoints) c ON r.route_id = c.route_id WHERE c.route_id IS NULL;

-- A5 no vehicle or driver double-booked (each must be 0)
SELECT COUNT(*) AS vehicle_overlaps FROM ref.ref_trips a JOIN ref.ref_trips b ON a.vehicle_id = b.vehicle_id AND a.trip_id < b.trip_id AND a.planned_start_time < b.planned_end_time AND b.planned_start_time < a.planned_end_time;
SELECT COUNT(*) AS driver_overlaps  FROM ref.ref_trips a JOIN ref.ref_trips b ON a.driver_id  = b.driver_id  AND a.trip_id < b.trip_id AND a.planned_start_time < b.planned_end_time AND b.planned_start_time < a.planned_end_time;

-- A6 cargo limits sane for temperature-controlled cargo (expect 0)
SELECT COUNT(*) AS bad_cargo_limits FROM ref.ref_cargo_types WHERE temperature_max_c IS NOT NULL AND temperature_min_c >= temperature_max_c;

-- A7 vehicle versions: two DIFFERENT rows with the same vehicle_id + updated_at (expect 0). A file loaded twice is fine, it is de-duplicated.
SELECT COUNT(*) AS conflicting_versions FROM (
  SELECT vehicle_id, updated_at FROM ref.ref_vehicles_raw GROUP BY vehicle_id, updated_at
  HAVING COUNT(DISTINCT concat_ws('|', vehicle_type, refrigeration_capable, device_id, registration_number)) > 1);
-- A7b how many vehicles have more than one version in the source (informational: this is your SCD test data; expect > 0)
SELECT COUNT(*) AS vehicles_with_changes FROM (SELECT vehicle_id FROM ref.ref_vehicles_raw GROUP BY vehicle_id HAVING COUNT(DISTINCT updated_at) > 1);

-- A8 cold cargo on a vehicle version that cannot refrigerate at planned start (expect 0, or exactly the number the generator injected on purpose)
-- (run after nb_gold_dims; same logic as fact_trip_performance.invalid_assignment)
SELECT t.trip_id, t.vehicle_id, t.cargo_type_id, t.planned_start_time
FROM ref.ref_trips t JOIN ref.ref_cargo_types c ON t.cargo_type_id = c.cargo_type_id AND c.temperature_max_c IS NOT NULL
JOIN gold.dim_vehicle v ON t.vehicle_id = v.vehicle_id AND t.planned_start_time >= v.effective_from AND t.planned_start_time < v.effective_to
WHERE NOT v.refrigeration_capable;

-- A9 business rules Gold reads that are missing (a missing rule silently uses the default in the notebook; expect 0 rows if you want zero surprises)
SELECT n AS missing_rule FROM (VALUES
 ('idle_speed_threshold_kmh'),('idle_minutes_threshold'),('speeding_tolerance_kmh'),('gps_accuracy_max_meters'),
 ('deviation_consecutive_readings'),('expected_interval_gps_sec'),('expected_interval_engine_sec'),('expected_interval_fuel_sec'),
 ('expected_interval_temp_sec'),('score_weight_speeding'),('score_weight_idling'),('score_weight_deviation'),('score_weight_lateness'),
 ('score_limit_speeding_pct'),('score_limit_idling_pct'),('score_limit_deviation_episodes'),('score_limit_late_minutes')) AS r(n)
WHERE n NOT IN (SELECT rule_name FROM ref.ref_business_rules);

StatementMeta(, 07465b6b-e9fa-4ee9-a12d-40e88cca092c, 16, Finished, Available, Finished, True)

<Spark SQL result set with 6 rows and 2 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

...

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 0 rows and 4 fields>

<Spark SQL result set with 0 rows and 1 fields>

In [ ]:
%%sql
SELECT *
FROM ref.ref_business_rules
WHERE rule_name LIKE '%lateness%';

StatementMeta(, 680f11e0-0c06-4ff5-bda1-9d303d1dc676, 21, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [2]:
mssparkutils.notebook.exit("done")

StatementMeta(, 93fc0106-adf7-49df-b6a8-128228b1189c, 4, Finished, Available, Finished, True)

In [ ]:
# Release cached data and clear Spark cache
spark.catalog.clearCache()